# Wildlife Re-Identification — three models

| Model | Approach | Output CSV | Kaggle public |
|---|---|---|---|
| 1 | CSLS + Jaccard retrieval, SIFT, query graph, two-stage LightGBM, 5-member ensemble | `model1_improved_ensemble.csv` | 0.86002 |
| 2 | XQDA + KDE / two-hop retrieval, validation-consistent features, 5-member ensemble | `model2_xqda_kde_ensemble.csv` | |
| 3 | V8 fusion: PCA + LDA retrieval, SIFT, two-stage LightGBM, 9-member ensemble | `model3_v8_fusion_ensemble.csv` | 0.85677 |

Only the provided features are used (MiewID vectors, SIFT keypoints/descriptors, the CSVs) with the allowed libraries; no images, pre-trained models or external data.

**How to run:** run the package cell below, then the models in order. Each model loads the data itself and keeps its own caches (Model 1: `Work/sift_rich_cache.pkl`, Model 2: `Work/m2_*.pkl`, Model 3: `Work/Model3_V8/`). A reset cell after Models 1 and 2 frees memory, so each model starts clean. To run a single model, run the package cell and then that model's cells. All three together take roughly 1.5–2.5 hours on a CPU (Model 1 took 25 minutes in its last run).

In [ ]:
import sys
import subprocess
import importlib.util

# Allowed competition libraries: (import_name, pip_package_name)
REQUIRED_PACKAGES = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("sklearn", "scikit-learn"),
    ("lightgbm", "lightgbm"),
    ("xgboost", "xgboost"),
    ("catboost", "catboost"),
    ("cv2", "opencv-python"),
    ("networkx", "networkx"),
]

missing = [pip_name for mod_name, pip_name in REQUIRED_PACKAGES if importlib.util.find_spec(mod_name) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installation complete.")
else:
    print("All required packages are already installed.")

## Model 1 — Anti-Overfitting High-MAP@5 Pipeline (Kaggle public 0.86002)

**Strictly compliant with all competition rules** (only `numpy`, `pandas`, `scipy`, `scikit-learn`, `lightgbm`, `xgboost`, `catboost`, `opencv-python`, `networkx`; only provided MiewID & SIFT features; no deep learning, no external data, no raw images).

### Key Upgrades Over the Previous Pipeline (0.85677 LB)
1. **Power-Law Regularized Whitening ($\alpha = 0.5$) + Multi-Space Embeddings**: Replaces noise-amplifying full PCA whitening (`whiten=True`) with eigenvalue power-whitening, combined with Regularized LDA and L2-normalized identity centroids.
2. **Hubness Reduction (CSLS) & k-Reciprocal Jaccard Reranking**: Pure distance-based retrieval features that eliminate high-dimensional hubness and dramatically improve both top-5 ranking and `new_individual` separation.
3. **Upgraded Geometric SIFT Verification (5-Feature Affine + Sanity Check)**: Faster pre-computed `float16` RootSIFT, sanity-checked `estimateAffine2D` (handles curved flanks while rejecting degenerate warps), plus **inlier ratio**, **mean inlier similarity**, and **spatial spread** (kills clustered background false matches).
4. **Full-Graph Stage-2 Transductive Propagation (`networkx`)**: Removes the 50% test-set split from the old `Stage2.py` (which threw away half of all test-to-test same-animal links) and uses thresholded graph components that are density-consistent between CV and Test.
5. **Anti-Overfitting Convex Log-Loss Blending + Per-Species Prior-Shift EM**: Replaces 18-parameter coordinate ascent on discrete MAP@5 with smooth simplex log-loss calibration and per-species Bayes prior-shift adjustment.

**Target Runtime**: ~4–8 minutes end-to-end on CPU (SIFT features auto-cache to `Work/sift_rich_cache.pkl` so re-runs take <2 minutes).

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import gc
import time
import pickle
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.optimize import minimize
from scipy.special import expit, logit

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRanker

warnings.filterwarnings("ignore")
cv2.setNumThreads(1)

T_START = time.time()
NEW = "new_individual"
SEED = 42
N_WORKERS = max(1, min(8, os.cpu_count() or 4))

# ---- Auto-detect data directory (works on Kaggle, Colab, and local Windows/Linux) ----
BASE_DIR = os.environ.get("REID_BASE", None)  # Optional manual override

def resolve_paths(base_override=None):
    candidates = []
    if base_override:
        candidates.append(Path(base_override))
    candidates += [
        Path.cwd(),
        Path.cwd() / "reid_student_data_main",
        Path.cwd() / "projectwild3",
        Path.cwd() / "projectwild3" / "reid_student_data_main",
        Path("/kaggle/input"),
    ]
    # Also search immediate subdirs of /kaggle/input if present
    ki = Path("/kaggle/input")
    if ki.exists():
        for p in ki.rglob("train.csv"):
            candidates.insert(0, p.parent)

    data_dir = None
    for c in candidates:
        if (c / "train.csv").exists() and (c / "miewid_features.npy").exists():
            data_dir = c
            break
        if (c / "reid_student_data_main" / "train.csv").exists():
            data_dir = c / "reid_student_data_main"
            break
    if data_dir is None:
        raise FileNotFoundError(
            "Could not locate train.csv and miewid_features.npy. "
            "Set BASE_DIR = '/path/to/your/folder' above."
        )

    def find_sift(name):
        for p in [data_dir / name, data_dir.parent / name, Path.cwd() / name]:
            if p.exists():
                return p
        # Search upward/siblings on Kaggle
        for p in data_dir.parent.rglob(name):
            return p
        raise FileNotFoundError(f"Could not locate {name}")

    work_dir = Path("/kaggle/working/Work") if Path("/kaggle/working").exists() else (Path.cwd() / "Work")
    work_dir.mkdir(parents=True, exist_ok=True)
    out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    return data_dir, find_sift("sift_features_train.npz"), find_sift("sift_features_test.npz"), work_dir, out_dir

DATA_DIR, SIFT_TRAIN_PATH, SIFT_TEST_PATH, WORK_DIR, OUT_DIR = resolve_paths(BASE_DIR)
print(f"DATA_DIR:   {DATA_DIR}")
print(f"SIFT_TRAIN: {SIFT_TRAIN_PATH}")
print(f"SIFT_TEST:  {SIFT_TEST_PATH}")
print(f"WORK_DIR:   {WORK_DIR} | WORKERS: {N_WORKERS}")

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
for df in (train, test):
    df["viewpoint"] = df["viewpoint"].fillna("unknown").astype(str)

all_meta = pd.concat([train[["id", "species", "viewpoint"]],
                      test[["id", "species", "viewpoint"]]], ignore_index=True)
SP_MAP = dict(zip(all_meta["id"], all_meta["species"]))
VP_MAP = dict(zip(all_meta["id"], all_meta["viewpoint"]))
LAB_MAP = dict(zip(train["id"], train["identity"]))
SPECIES_LIST = sorted(train["species"].unique().tolist())

F_RAW = np.load(DATA_DIR / "miewid_features.npy").astype(np.float32)
MIEW_IDS = pd.read_csv(DATA_DIR / "miewid_ids.csv")["id"].tolist()
assert len(MIEW_IDS) == len(F_RAW), "miewid_ids.csv and miewid_features.npy mismatch!"
ROW_MAP = {img_id: idx for idx, img_id in enumerate(MIEW_IDS)}
F_UNIT = F_RAW / (np.linalg.norm(F_RAW, axis=1, keepdims=True) + 1e-12)

SIDES = {
    "left": "L", "frontleft": "L", "backleft": "L",
    "right": "R", "frontright": "R", "backright": "R",
}
SIDE_CODE = {"L": 0, "R": 1, "O": 2, "U": 3}

def get_side(vp):
    return "U" if vp == "unknown" else SIDES.get(vp, "O")

def view_relation(vp_a, vp_b):
    # 0: same flank, 1: flank vs other, 2: opposite flanks (L vs R), 3: unknown
    sa, sb = get_side(vp_a), get_side(vp_b)
    if "U" in (sa, sb):
        return 3
    if sa == sb:
        return 0
    return 2 if {sa, sb} == {"L", "R"} else 1

def map5_score(truth_dict, preds_dict):
    tot = 0.0
    for q, t in truth_dict.items():
        for k, lab in enumerate(preds_dict.get(q, [])[:5]):
            if lab == t:
                tot += 1.0 / (k + 1)
                break
    return tot / max(len(truth_dict), 1)

def check_and_save_submission(preds_dict, filename="submission.csv"):
    sp_ident = dict(zip(train["identity"], train["species"]))
    sp_test = dict(zip(test["id"], test["species"]))
    rows = []
    for img_id in test["id"]:
        labs = []
        for l in preds_dict.get(img_id, []):
            if l not in labs and (l == NEW or sp_ident.get(l) == sp_test[img_id]):
                labs.append(l)
        if not labs:
            labs = [NEW]
        rows.append((img_id, " ".join(labs[:5])))
    sub = pd.DataFrame(rows, columns=["id", "predictions"])
    assert len(sub) == len(test) and set(sub["id"]) == set(test["id"])
    assert not sub["id"].duplicated().any()
    out_path = OUT_DIR / filename
    sub.to_csv(out_path, index=False)
    print(f"Saved valid submission -> {out_path} ({len(sub)} rows)")
    return sub

print(f"Train: {len(train)} images, {train['identity'].nunique()} identities | Test: {len(test)} images")
print("Species breakdown (train):", train["species"].value_counts().to_dict())
print("Species breakdown (test): ", test["species"].value_counts().to_dict())

### 1.1 Power-Law Whitening & Encounter-Aware Disjoint 5-Fold Validation

* **Why Power-Whitening ($\alpha = 0.5$)?** Full PCA whitening (`whiten=True`, $\alpha=1.0$) divides by $\sqrt{\lambda_i}$, which amplifies tail noise in components 200–512. Power-whitening divides by $(\lambda_i + \epsilon)^{\alpha/2}$ with $\alpha=0.5$, dampening burst/background dominance while keeping discriminative fine-grained features clean.
* **Why Encounter-Aware Disjoint 5-Fold CV?** Each identity's near-duplicate burst photos (connected components at cosine > 0.80) are held out *together*, and each identity is held out as `new_individual` in **exactly one** of the 5 folds. This prevents duplicate-query inflation across folds and mirrors the true hidden test set.

In [ ]:
def compute_power_whitened_pca(F, row_map, sp_map, n_comp=512, alpha=0.5, eps=1e-4):
    """Per-species centered PCA with eigenvalue power-whitening: z / (var + eps)^(alpha/2)."""
    ids = sorted(row_map, key=row_map.get)
    spa = np.array([sp_map[i] for i in ids])
    E = np.zeros((len(F), n_comp), dtype=np.float32)
    for s in SPECIES_LIST:
        m = (spa == s)
        Z = F[m].astype(np.float64)
        Z = Z - Z.mean(axis=0, keepdims=True)
        k = min(n_comp, Z.shape[0] - 1, Z.shape[1])
        pca = PCA(n_components=k, svd_solver="randomized", random_state=SEED)
        U = pca.fit_transform(Z)
        scale = np.power(pca.explained_variance_ + eps, -0.5 * alpha)
        U = U * scale
        E[m, :k] = U.astype(np.float32)
    E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    return E

def build_encounters(train_df, W, row_map, tau=0.80):
    """Connected components of same-identity photos with power-whitened cosine > tau."""
    enc = {}
    for ident, g in train_df.groupby("identity"):
        ids = g["id"].tolist()
        if len(ids) == 1:
            enc[ids[0]] = f"{ident}#0"
            continue
        X = W[[row_map[i] for i in ids]]
        adj = csr_matrix((X @ X.T) > tau)
        _, comp = connected_components(adj, directed=False)
        for img_id, c in zip(ids, comp):
            enc[img_id] = f"{ident}#{c}"
    return enc

def make_disjoint_folds(train_df, enc_map, n_folds=5, cap=10, seed=SEED):
    """
    Strictly disjoint 5-fold open-set validation:
    - Every identity is hidden as `new_individual` in exactly 1 fold (~20% unknown rate).
    - Known identities with >=2 encounters have disjoint encounters assigned across the remaining folds.
    """
    rng = np.random.default_rng(seed)
    lab = dict(zip(train_df["id"], train_df["identity"]))
    unk_fold = {}
    for _, d in train_df.groupby("species"):
        idents = rng.permutation(d["identity"].unique())
        for idx, ident in enumerate(idents):
            unk_fold[ident] = idx % n_folds

    # Pre-assign known encounters to folds != unk_fold[ident]
    enc_fold = {}
    for ident, g in train_df.groupby("identity"):
        uf = unk_fold[ident]
        encs = rng.permutation(sorted({enc_map[i] for i in g["id"]}))
        if len(encs) >= 2:
            avail_folds = [f for f in range(n_folds) if f != uf]
            # Leave at least 1 encounter always in gallery when that fold is queried
            for idx, e in enumerate(encs):
                enc_fold[e] = avail_folds[idx % len(avail_folds)]

    folds = []
    for f in range(n_folds):
        queries, hidden = [], set()
        for ident, g in train_df.groupby("identity"):
            ids = g["id"].to_numpy()
            if unk_fold[ident] == f:
                hidden.update(ids)
                q_pick = rng.choice(ids, min(cap, len(ids)), replace=False)
                queries.extend(q_pick.tolist())
            else:
                qi = [i for i in ids if enc_fold.get(enc_map[i], -1) == f]
                # Ensure at least 1 photo remains in gallery for this known identity
                if 0 < len(qi) < len(ids):
                    hidden.update(qi)
                    q_pick = rng.choice(qi, min(cap, len(qi)), replace=False)
                    queries.extend(q_pick.tolist())
        qset = set(queries)
        gallery = [i for i in train_df["id"] if i not in qset and i not in hidden]
        gal_idents = {lab[i] for i in gallery}
        truth = {q: (lab[q] if lab[q] in gal_idents else NEW) for q in sorted(qset)}
        folds.append({"fold": f, "queryIds": sorted(qset), "galleryIds": gallery, "truth": truth})
    return folds

W_PWR = compute_power_whitened_pca(F_RAW, ROW_MAP, SP_MAP, n_comp=512, alpha=0.50)
W_FULL = compute_power_whitened_pca(F_RAW, ROW_MAP, SP_MAP, n_comp=512, alpha=0.85)
ENC_MAP = build_encounters(train, W_PWR, ROW_MAP, tau=0.80)
FOLDS = make_disjoint_folds(train, ENC_MAP, n_folds=5)

print(f"Found {len(set(ENC_MAP.values()))} encounters across {len(train)} train photos.")
for f in FOLDS:
    t = pd.Series(f["truth"])
    print(f"  Fold {f['fold']}: {len(t):4d} queries | new={np.mean(t == NEW):.1%} | gallery={len(f['galleryIds']):4d}")

### 1.2 Multi-Space Retrieval, Hubness Reduction (CSLS) & k-Reciprocal Jaccard Reranking

For each fold (strictly fitting LDA **only on that fold's gallery**) and for the full test set:
1. **Regularized LDA** ($d=128$, `shrinkage='auto'`) fitted on `W_PWR` using gallery identities with $\ge 2$ photos (excluding singletons prevents zero-variance covariance distortion), blended 50/50 with `W_PWR`.
2. **CSLS (Cross-Domain Similarity Local Scaling)**: $\text{CSLS}(q, g) = 2\cos(q,g) - \bar{s}_{10}(q) - \bar{s}_{10}(g)$, where $\bar{s}_{10}(g)$ is gallery image $g$'s average cosine to its top-10 other-identity neighbors (hubness penalty!).
3. **k-Reciprocal Jaccard Similarity**: Measures mutual neighborhood overlap between query $q$ and gallery image $g$ inside the species' combined similarity graph — a classical Re-ID reranking signal that is 100% k-NN / distance-based.
4. **Identity Centroids & Viewpoint-Aware Stats**: Also computes cosine to each identity's L2-normalized mean vector (`CentroidCos`, `SameSideCentroidCos`) and query-normalized Z-scores (`CosZ`).

In [ ]:
TOP_ID = 35
RAW_SLOTS = 7
TOP_IMG_BASE = 2
TOP_IMG_DEEP_K = 5
TOP_IMG_DEEP_CAP = 8

def fit_gallery_lda_blend(W, row_map, sp_map, lab_map, gallery_ids, n_comp=128, mix=0.50):
    ids = sorted(row_map, key=row_map.get)
    spa = np.array([sp_map[i] for i in ids])
    L = np.zeros((len(W), n_comp), dtype=np.float32)
    for s in SPECIES_LIST:
        g = [i for i in gallery_ids if sp_map[i] == s]
        y = np.array([lab_map[i] for i in g])
        # Use identities with >= 2 images for cleaner within-class scatter estimation
        vc = pd.Series(y).value_counts()
        valid_ids = set(vc[vc >= 2].index)
        g_fit = [i for i in g if lab_map[i] in valid_ids]
        if len(valid_ids) < 3:
            g_fit = g
        y_fit = np.array([lab_map[i] for i in g_fit])
        k = min(n_comp, len(np.unique(y_fit)) - 1, W.shape[1])
        lda = LinearDiscriminantAnalysis(n_components=k, solver="eigen", shrinkage="auto")
        lda.fit(W[[row_map[i] for i in g_fit]], y_fit)
        L[spa == s, :k] = lda.transform(W[spa == s]).astype(np.float32)
    L /= (np.linalg.norm(L, axis=1, keepdims=True) + 1e-12)
    E = np.hstack([np.sqrt(1.0 - mix) * W, np.sqrt(mix) * L]).astype(np.float32)
    E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    return E

def compute_k_reciprocal_jaccard(E_q, E_g, k1=15, k2=5):
    """
    Fast sparse k-reciprocal Jaccard similarity between queries (nq) and gallery (ng).
    Pure k-NN distance-based reranking (Zhong et al., CVPR 2017).
    """
    X = np.vstack([E_q, E_g]).astype(np.float32)
    N = len(X)
    nq = len(E_q)
    S = X @ X.T
    # Top-k1 neighbors for every node
    k1_eff = min(k1, N - 1)
    top_idx = np.argpartition(-S, k1_eff, axis=1)[:, :k1_eff + 1]
    # Build boolean membership matrix for fast reciprocal check
    rows = np.repeat(np.arange(N), k1_eff + 1)
    cols = top_idx.ravel()
    vals = np.exp(np.clip(S[rows, cols] * 5.0, -20, 20)).astype(np.float32)
    A = csr_matrix((vals, (rows, cols)), shape=(N, N))
    # Mutual (reciprocal) weights: elementwise min of A and A^T
    R = A.minimum(A.T).tocsr()
    # Local query expansion (average over top-k2 neighbors)
    k2_eff = min(k2, N - 1)
    top_k2 = np.argpartition(-S, k2_eff, axis=1)[:, :k2_eff + 1]
    r2 = np.repeat(np.arange(N), k2_eff + 1)
    c2 = top_k2.ravel()
    Qexp = csr_matrix((np.full(len(r2), 1.0 / (k2_eff + 1), dtype=np.float32), (r2, c2)), shape=(N, N))
    V = (Qexp @ R).tocsr()
    # L1-normalize rows so generalized Jaccard = sum(min(v_q, v_g)) / (2 - sum(min))
    # Fast cosine approximation on L2-normalized reciprocal feature vectors
    row_norms = np.sqrt(np.asarray(V.multiply(V).sum(axis=1)).ravel()) + 1e-12
    V = V.multiply(1.0 / row_norms[:, None]).tocsr()
    J = (V[:nq] @ V[nq:].T).toarray().astype(np.float32)
    return J

def retrieve_candidates(E, W_pwr, W_full, F_unit, row_map, sp_map, vp_map, query_ids, gallery_df):
    out_rows = []
    for s, G in gallery_df.groupby("species"):
        qids = [q for q in query_ids if sp_map[q] == s]
        if not qids:
            continue
        G = G.sort_values("identity").reset_index(drop=True)
        gid = G["id"].to_numpy()
        glab = G["identity"].to_numpy()
        gside = np.array([get_side(vp_map[g]) for g in gid])
        starts = np.r_[0, np.flatnonzero(glab[1:] != glab[:-1]) + 1]
        counts = np.diff(np.r_[starts, len(glab)])
        seg = np.repeat(np.arange(len(starts)), counts)
        labs = glab[starts]
        n_id = len(starts)

        q_idx = [row_map[q] for q in qids]
        g_idx = [row_map[g] for g in gid]

        Eq, Eg = E[q_idx], E[g_idx]
        Wq, Wg = W_pwr[q_idx], W_pwr[g_idx]
        WFq, WFg = W_full[q_idx], W_full[g_idx]
        Rq, Rg = F_unit[q_idx], F_unit[g_idx]

        # Identity centroids in main blended space E
        centroids = np.zeros((n_id, Eg.shape[1]), dtype=np.float32)
        for j in range(n_id):
            c_vec = Eg[starts[j]:starts[j] + counts[j]].mean(axis=0)
            centroids[j] = c_vec / (np.linalg.norm(c_vec) + 1e-12)
        S_cent = Eq @ centroids.T  # (nq, n_id)

        # Gallery-side hubness: mean top-10 similarity of each gallery image to OTHER identities
        S_gg = Eg @ Eg.T
        same_id_mask = (seg[:, None] == seg[None, :])
        S_gg[same_id_mask] = -1.0
        k_hub = min(10, S_gg.shape[1] - 1)
        g_hub = np.partition(S_gg, -k_hub, axis=1)[:, -k_hub:].mean(axis=1)  # (ng,)

        # Main similarity matrix + CSLS + k-Reciprocal Jaccard
        S = Eq @ Eg.T  # (nq, ng)
        q_hub = np.partition(S, -k_hub, axis=1)[:, -k_hub:].mean(axis=1)     # (nq,)
        S_csls = 2.0 * S - q_hub[:, None] - g_hub[None, :]
        J_mat = compute_k_reciprocal_jaccard(Eq, Eg, k1=15, k2=5)

        # Query-level background mean/std for Z-score normalization
        q_mean = S.mean(axis=1)
        q_std = S.std(axis=1) + 1e-6

        # Per-identity maxima in each space
        M_main = np.maximum.reduceat(S, starts, axis=1)
        M_csls = np.maximum.reduceat(S_csls, starts, axis=1)
        M_jac = np.maximum.reduceat(J_mat, starts, axis=1)
        M_raw = np.maximum.reduceat(Rq @ Rg.T, starts, axis=1)
        M_wp = np.maximum.reduceat(Wq @ Wg.T, starts, axis=1)
        M_wf = np.maximum.reduceat(WFq @ WFg.T, starts, axis=1)

        R_raw = np.argsort(np.argsort(-M_raw, axis=1), axis=1)
        R_wp = np.argsort(np.argsort(-M_wp, axis=1), axis=1)
        R_csls = np.argsort(np.argsort(-M_csls, axis=1), axis=1)
        R_jac = np.argsort(np.argsort(-M_jac, axis=1), axis=1)

        # Combined retrieval score for candidate selection (CSLS + Jaccard)
        M_sel = M_csls + 0.35 * M_jac

        for r, q in enumerate(qids):
            main_ord = np.argsort(-M_sel[r])
            order = main_ord[:TOP_ID - RAW_SLOTS]
            chosen = set(order.tolist())
            raw_extra = [j for j in np.argsort(-M_raw[r]) if j not in chosen][:RAW_SLOTS]
            order = np.r_[order, raw_extra].astype(int)

            top15_g = np.argsort(-S_csls[r])[:15]
            knn15 = np.bincount(seg[top15_g], minlength=n_id)
            qs = get_side(vp_map[q])

            top1_cos = float(M_main[r].max())
            top2_cos = float(np.partition(M_main[r], -2)[-2]) if n_id > 1 else -1.0
            top1_csls = float(M_csls[r].max())
            top2_csls = float(np.partition(M_csls[r], -2)[-2]) if n_id > 1 else -1.0
            top1_jac = float(M_jac[r].max())
            top2_jac = float(np.partition(M_jac[r], -2)[-2]) if n_id > 1 else 0.0

            for rk, j in enumerate(order):
                a, n = starts[j], counts[j]
                v_cos = S[r, a:a + n]
                v_csls = S_csls[r, a:a + n]
                v_jac = J_mat[r, a:a + n]
                v_hub = g_hub[a:a + n]
                o = np.argsort(-v_csls)

                cap_img = TOP_IMG_DEEP_CAP if rk < TOP_IMG_DEEP_K else TOP_IMG_BASE
                best_imgs = list(gid[a + o[:cap_img]])

                gs = gside[a:a + n]
                same_idx = np.flatnonzero(gs == qs) if qs in "LR" else np.array([], dtype=int)
                opp_idx = np.flatnonzero(gs == {"L": "R", "R": "L"}.get(qs, "-"))
                oth_idx = np.flatnonzero(gs == "O")

                if len(same_idx):
                    bs = gid[a + same_idx[np.argmax(v_csls[same_idx])]]
                    if bs not in best_imgs:
                        best_imgs.append(bs)

                same_max = float(v_cos[same_idx].max()) if len(same_idx) else np.nan
                same_csls = float(v_csls[same_idx].max()) if len(same_idx) else np.nan
                opp_max = float(v_cos[opp_idx].max()) if len(opp_idx) else np.nan
                oth_max = float(v_cos[oth_idx].max()) if len(oth_idx) else np.nan
                n_sides = len(set(gs) - {"U"})

                if qs in "LR":
                    n_comp = int((gs == qs).sum())
                elif qs == "O":
                    n_comp = int(sum(vp_map[g] == vp_map[q] for g in gid[a:a + n]))
                else:
                    n_comp = -1

                out_rows.append((
                    q, labs[j],
                    float(v_cos[o[0]]),
                    float(v_cos[o[1]]) if n > 1 else np.nan,
                    float(v_cos[o[:3]].mean()),
                    float(v_csls[o[0]]),
                    float(v_csls[o[1]]) if n > 1 else np.nan,
                    float(v_csls[o[:3]].mean()),
                    float(v_jac[o[0]]),
                    float(S_cent[r, j]),
                    float((v_cos[o[0]] - q_mean[r]) / q_std[r]),
                    float(v_hub[o[0]]),
                    float(q_hub[r]),
                    int(knn15[j]),
                    same_max, same_csls, opp_max, oth_max, n_sides,
                    rk, int(R_csls[r, j]), int(R_jac[r, j]),
                    top1_cos, top1_cos - top2_cos,
                    top1_csls, top1_csls - top2_csls,
                    top1_jac, top1_jac - top2_jac,
                    int(n), n_comp,
                    float(M_raw[r, j]), int(R_raw[r, j]),
                    float(M_wp[r, j]), int(R_wp[r, j]),
                    float(M_wf[r, j]),
                    best_imgs,
                ))

    cols = [
        "Query", "Identity",
        "CosMax", "Cos2nd", "CosMean3",
        "CSLSMax", "CSLS2nd", "CSLSMean3",
        "JacMax", "CentroidCos", "CosZ", "GalHub", "QHub",
        "Knn15", "SameSideMax", "SameSideCSLS", "OppSideMax", "OtherViewMax", "NSides",
        "IdRank", "CSLSRank", "JacRank",
        "QTop1Cos", "QGap12Cos", "QTop1CSLS", "QGap12CSLS", "QTop1Jac", "QGap12Jac",
        "NGal", "NCompGal",
        "RawCosMax", "RawRank", "WPCosMax", "WPRank", "WFCosMax",
        "BestImgs",
    ]
    return pd.DataFrame(out_rows, columns=cols)

t0 = time.time()
tr_sub = train[["id", "identity", "species"]]
FOLD_EMBS = {}
CAND_FOLDS = []

for f in FOLDS:
    E_f = fit_gallery_lda_blend(W_PWR, ROW_MAP, SP_MAP, LAB_MAP, f["galleryIds"])
    FOLD_EMBS[f["fold"]] = E_f
    gal_df = tr_sub[tr_sub["id"].isin(set(f["galleryIds"]))]
    cf = retrieve_candidates(E_f, W_PWR, W_FULL, F_UNIT, ROW_MAP, SP_MAP, VP_MAP, f["queryIds"], gal_df)
    cf["Fold"] = f["fold"]
    cf["QKey"] = f"{f['fold']}:" + cf["Query"]
    cf["Truth"] = cf["Query"].map(f["truth"])
    CAND_FOLDS.append(cf)

E_TEST = fit_gallery_lda_blend(W_PWR, ROW_MAP, SP_MAP, LAB_MAP, train["id"].tolist())
CAND_TEST = retrieve_candidates(E_TEST, W_PWR, W_FULL, F_UNIT, ROW_MAP, SP_MAP, VP_MAP, test["id"].tolist(), tr_sub)
CAND_TEST["QKey"] = CAND_TEST["Query"]

V_ALL = pd.concat(CAND_FOLDS, ignore_index=True)

# Diagnostic: Candidate Recall@35 and Raw Retrieval Baseline MAP@5
recalls = []
for f, cf in zip(FOLDS, CAND_FOLDS):
    c_map = cf.groupby("Query")["Identity"].apply(set).to_dict()
    known_q = [q for q, t in f["truth"].items() if t != NEW]
    recalls.append(np.mean([f["truth"][q] in c_map.get(q, set()) for q in known_q]))
print(f"Retrieval complete in {time.time() - t0:.1f}s | Mean Recall@{TOP_ID} (known): {np.mean(recalls):.4f}")

### 1.3 Fast Geometric SIFT Verification (5 Rich Features + Affine Sanity Check)

Improvements over the previous `P3Sift.py`:
* **3–4× Faster**: Pre-computes `float16` RootSIFT once per unique image (instead of re-computing per pair inside the thread pool) and focuses SIFT budget on the highest-leverage candidate pairs + query-to-query graph edges.
* **5 Geometric Features per Pair**: `(nGood, nInliers, inlierRatio, meanInlierSim, spatialSpread)`.
* **Curved-Flank Affine + Degeneracy Guard**: Uses `cv2.estimateAffine2D` (handles 3D body curvature/foreshortening), verifies `det > 0` and reasonable scale/shear, and falls back to `estimateAffinePartial2D` if degenerate.
* **Auto-Cached**: Saved to `Work/sift_rich_cache.pkl` so subsequent runs load instantly.

In [ ]:
SIFT_RATIO = 0.80
RANSAC_PX = 14.0
Q_NBR_K = 10

def get_query_neighbors(E, row_map, qids, sp_map, k=Q_NBR_K):
    """Top-k most similar other queries of the same species (NO test-set splitting!)."""
    out = {}
    for s in SPECIES_LIST:
        qs = [q for q in qids if sp_map[q] == s]
        if len(qs) <= 1:
            continue
        X = E[[row_map[q] for q in qs]]
        S = X @ X.T
        np.fill_diagonal(S, -9.0)
        kk = min(k, len(qs) - 1)
        idx = np.argpartition(-S, kk, axis=1)[:, :kk]
        for a, q in enumerate(qs):
            nbrs = idx[a]
            nbrs = nbrs[np.argsort(-S[a, nbrs])]
            out[q] = [(qs[b], float(S[a, b])) for b in nbrs]
    return out

FOLD_QNBRS = {f["fold"]: get_query_neighbors(FOLD_EMBS[f["fold"]], ROW_MAP, f["queryIds"], SP_MAP) for f in FOLDS}
TEST_QNBRS = get_query_neighbors(E_TEST, ROW_MAP, test["id"].tolist(), SP_MAP)

# Collect all required SIFT pairs: candidate pairs + query-query neighbor pairs (top 6 neighbors)
req_pairs = set()
for df_c in [V_ALL, CAND_TEST]:
    # Only SIFT-verify top-25 candidates per query to stay well under 10 min runtime
    sub_c = df_c[df_c["IdRank"] < 25]
    for q, imgs in zip(sub_c["Query"], sub_c["BestImgs"]):
        for g in imgs:
            req_pairs.add((q, g))

for nb_dict in list(FOLD_QNBRS.values()) + [TEST_QNBRS]:
    for q, lst in nb_dict.items():
        for q2, sim in lst[:6]:
            if sim > 0.35:
                req_pairs.add((q, q2) if q < q2 else (q2, q))

print(f"Total unique SIFT pairs requested: {len(req_pairs):,}")

SIFT_CACHE_PATH = WORK_DIR / "sift_rich_cache.pkl"
NKP_CACHE_PATH = WORK_DIR / "nkp_cache.pkl"

if SIFT_CACHE_PATH.exists() and NKP_CACHE_PATH.exists():
    with open(SIFT_CACHE_PATH, "rb") as fh:
        SIFT_CACHE = pickle.load(fh)
    with open(NKP_CACHE_PATH, "rb") as fh:
        NKP_MAP = pickle.load(fh)
else:
    SIFT_CACHE, NKP_MAP = {}, {}

todo_pairs = [p for p in req_pairs if p not in SIFT_CACHE]
print(f"Cached: {len(SIFT_CACHE):,} | To compute: {len(todo_pairs):,}")

def load_sift_store(img_ids):
    """Loads SIFT xy (float32) and RootSIFT desc (float16) for requested images."""
    train_set = set(train["id"])
    npz_handles = {}
    empty = (np.zeros((0, 2), np.float32), np.zeros((0, 128), np.float16))
    store = {}
    for img_id in img_ids:
        src = SIFT_TRAIN_PATH if img_id in train_set else SIFT_TEST_PATH
        xy, d = None, None
        if src.is_dir():
            fx, fd = src / f"{img_id}_xy.npy", src / f"{img_id}_desc.npy"
            if fx.exists() and fd.exists():
                xy, d = np.load(fx), np.load(fd)
        else:
            if src not in npz_handles:
                npz_handles[src] = np.load(src)
            z = npz_handles[src]
            kx, kd = f"{img_id}_xy", f"{img_id}_desc"
            if kx in z.files and kd in z.files:
                xy, d = z[kx], z[kd]
        if xy is None or len(d) == 0:
            store[img_id] = empty
        else:
            d = d.astype(np.float32).reshape(-1, 128)
            d /= (d.sum(axis=1, keepdims=True) + 1e-7)
            d = np.sqrt(d).astype(np.float16)
            store[img_id] = (xy.astype(np.float32).reshape(-1, 2), d)
    for h in npz_handles.values():
        h.close()
    return store

def match_pair_rich(qxy, qd32, gxy, gd16):
    """
    Returns (nGood, nInliers, inlierRatio, meanInlierSim, spatialSpread).
    """
    if len(qd32) < 2 or len(gd16) < 2:
        return (0, 0, 0.0, 0.0, 0.0)
    gd32 = gd16.astype(np.float32)
    S = qd32 @ gd32.T  # unit RootSIFT cosine
    r = np.arange(len(S))
    j1 = S.argmax(axis=1)
    s1 = S[r, j1].copy()
    S[r, j1] = -np.inf
    s2 = S.max(axis=1)
    d1 = np.sqrt(np.maximum(2.0 - 2.0 * s1, 0.0))
    d2 = np.sqrt(np.maximum(2.0 - 2.0 * s2, 0.0))
    qi = np.flatnonzero(d1 < SIFT_RATIO * d2)
    if len(qi) == 0:
        return (0, 0, 0.0, 0.0, 0.0)
    gj = j1[qi]
    # One-to-one deduplication (keep best match per gallery keypoint)
    o = np.argsort(d1[qi])
    qi, gj = qi[o], gj[o]
    _, u = np.unique(gj, return_index=True)
    qi, gj = qi[u], gj[u]
    n_good = int(len(qi))
    if n_good < 4:
        return (n_good, 0, 0.0, 0.0, 0.0)

    pts_q, pts_g = qxy[qi], gxy[gj]
    M, inl = cv2.estimateAffine2D(
        pts_q, pts_g, method=cv2.RANSAC,
        ransacReprojThreshold=RANSAC_PX, maxIters=1500, confidence=0.99
    )
    use_partial = True
    if M is not None and inl is not None:
        A = M[:, :2]
        det = float(np.linalg.det(A))
        sv = np.linalg.svd(A, compute_uv=False)
        if det > 0.05 and sv[1] > 0.2 and sv[0] < 4.5 and (sv[0] / (sv[1] + 1e-6)) < 3.5:
            use_partial = False

    if use_partial:
        _, inl = cv2.estimateAffinePartial2D(
            pts_q, pts_g, method=cv2.RANSAC,
            ransacReprojThreshold=RANSAC_PX, maxIters=1500, confidence=0.99
        )

    if inl is None:
        return (n_good, 0, 0.0, 0.0, 0.0)

    mask = inl.ravel().astype(bool)
    n_inl = int(mask.sum())
    if n_inl < 3:
        return (n_good, n_inl, float(n_inl / n_good), 0.0, 0.0)

    inlier_sims = s1[qi[mask]]
    mean_sim = float(inlier_sims.mean())
    # Spatial spread of inliers on query image (normalized by overall query keypoint spread)
    q_span = float(np.hypot(qxy[:, 0].std(), qxy[:, 1].std()) + 1e-5)
    inl_pts = pts_q[mask]
    inl_spread = float(np.hypot(inl_pts[:, 0].std(), inl_pts[:, 1].std()) / q_span)
    return (n_good, n_inl, float(n_inl / n_good), mean_sim, inl_spread)

if todo_pairs:
    t0 = time.time()
    need_ids = sorted({i for p in todo_pairs for i in p})
    sift_store = load_sift_store(need_ids)
    for img_id, (_, d) in sift_store.items():
        NKP_MAP[img_id] = len(d)
    print(f"Loaded & converted RootSIFT for {len(need_ids):,} images in {time.time() - t0:.1f}s")

    by_q = {}
    for q, g in todo_pairs:
        by_q.setdefault(q, []).append(g)

    def _worker(item):
        q, gs = item
        qxy, qd16 = sift_store[q]
        qd32 = qd16.astype(np.float32)
        return [((q, g), match_pair_rich(qxy, qd32, sift_store[g][0], sift_store[g][1])) for g in gs]

    t1, done = time.time(), 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        for res in ex.map(_worker, by_q.items(), chunksize=12):
            SIFT_CACHE.update(res)
            done += len(res)
            if done % 40000 < len(res):
                rate = done / max(time.time() - t1, 1e-3)
                print(f"  SIFT progress: {done:,}/{len(todo_pairs):,} ({rate:.0f} pairs/s)")

    with open(SIFT_CACHE_PATH, "wb") as fh:
        pickle.dump(SIFT_CACHE, fh)
    with open(NKP_CACHE_PATH, "wb") as fh:
        pickle.dump(NKP_MAP, fh)
    del sift_store
    gc.collect()
    print(f"SIFT matching finished in {time.time() - t1:.1f}s -> cached to {SIFT_CACHE_PATH}")

### 1.4 Feature Engineering (Multi-Space + Hubness + Rich Comparable-View SIFT)

Separates *"matched SIFT on a comparable view and failed"* from *"never had a comparable-view photo in gallery"*, and adds query-relative margins across CSLS, Jaccard, Raw cosine, and geometric SIFT inliers/spread.

In [ ]:
def second_largest(s):
    s = s.dropna()
    return s.nlargest(2).iloc[-1] if len(s) >= 2 else np.nan

def add_rich_features(df, sift_cache, nkp_map, vp_map, sp_map):
    c = df.copy()
    n = len(c)
    inl_max = np.full(n, np.nan, dtype=np.float32)
    good_max = np.full(n, np.nan, dtype=np.float32)
    inl_sum = np.full(n, np.nan, dtype=np.float32)
    inl_cnt8 = np.full(n, np.nan, dtype=np.float32)
    ratio_max = np.full(n, np.nan, dtype=np.float32)
    sim_max = np.full(n, np.nan, dtype=np.float32)
    spread_max = np.full(n, np.nan, dtype=np.float32)
    nkp_g = np.full(n, np.nan, dtype=np.float32)

    n_comp_chk = np.full(n, np.nan, dtype=np.float32)
    comp_inl_max = np.full(n, np.nan, dtype=np.float32)
    comp_inl_norm = np.full(n, np.nan, dtype=np.float32)
    comp_spread = np.full(n, np.nan, dtype=np.float32)
    noncomp_inl = np.full(n, np.nan, dtype=np.float32)
    vrel = np.zeros(n, dtype=np.int8)

    queries = c["Query"].tolist()
    best_imgs_list = c["BestImgs"].tolist()

    for i, (q, imgs) in enumerate(zip(queries, best_imgs_list)):
        vrel[i] = view_relation(vp_map[q], vp_map[imgs[0]])
        nkp_g[i] = nkp_map.get(imgs[0], np.nan)
        qs = get_side(vp_map[q])
        nkq_i = nkp_map.get(q, 0)

        vals = []
        comp_vals, noncomp_vals = [], []
        for g in imgs:
            res = sift_cache.get((q, g))
            if res is None:
                continue
            vals.append(res)
            same = (get_side(vp_map[g]) == qs) if qs in "LR" else (vp_map[g] == vp_map[q] if qs == "O" else False)
            if same:
                comp_vals.append((res[1], res[1] / (min(nkq_i, nkp_map.get(g, 0)) + 1.0), res[4]))
            else:
                noncomp_vals.append(res[1])

        if vals:
            inl_max[i] = max(v[1] for v in vals)
            good_max[i] = max(v[0] for v in vals)
            inl_sum[i] = sum(v[1] for v in vals)
            inl_cnt8[i] = sum(v[1] >= 8 for v in vals)
            # Take ratio/sim/spread from the best-inlier image
            best_v = max(vals, key=lambda v: (v[1], v[3]))
            ratio_max[i] = best_v[2]
            sim_max[i] = best_v[3]
            spread_max[i] = best_v[4]

        if qs != "U":
            n_comp_chk[i] = len(comp_vals)
        if comp_vals:
            best_cv = max(comp_vals, key=lambda x: x[0])
            comp_inl_max[i] = best_cv[0]
            comp_inl_norm[i] = best_cv[1]
            comp_spread[i] = best_cv[2]
        if noncomp_vals:
            noncomp_inl[i] = max(noncomp_vals)

    c["InlMax"] = inl_max
    c["GoodMax"] = good_max
    c["InlSum"] = inl_sum
    c["InlCnt8"] = inl_cnt8
    c["InlRatio"] = ratio_max
    c["InlMeanSim"] = sim_max
    c["InlSpread"] = spread_max
    c["InlQuality"] = np.log1p(np.nan_to_num(inl_max, nan=0.0)) * np.nan_to_num(spread_max, nan=0.0)
    c["NChecked"] = c["BestImgs"].str.len()

    c["NCompChecked"] = n_comp_chk
    c["CompInlMax"] = comp_inl_max
    c["CompInlNorm"] = comp_inl_norm
    c["CompSpread"] = comp_spread
    c["NonCompInlMax"] = noncomp_inl
    c["NoCompView"] = np.where(c["NCompGal"] < 0, np.nan, (c["NCompGal"] == 0).astype(np.float32))
    c["LogNCompGal"] = np.log1p(c["NCompGal"].clip(lower=0))

    # Query-relative aggregations
    g_inl = c.groupby("QKey")["InlMax"]
    c["QInlMax"] = g_inl.transform("max")
    c["QInl2nd"] = g_inl.transform(second_largest)
    c["InlRank"] = g_inl.rank(ascending=False, method="min")
    c["InlFrac"] = c["InlMax"] / (c["QInlMax"] + 1.0)
    c["InlMargin"] = c["InlMax"] - np.where(c["InlMax"] >= c["QInlMax"], c["QInl2nd"], c["QInlMax"])
    c["QN12"] = (c["InlMax"] >= 12).groupby(c["QKey"]).transform("sum")
    c["QCompInlMax"] = c.groupby("QKey")["CompInlMax"].transform("max")

    nkq = c["Query"].map(nkp_map).astype(np.float32)
    c["InlNorm"] = c["InlMax"] / (np.minimum(nkq, nkp_g) + 1.0)
    c["LogNkpQ"] = np.log1p(nkq)
    c["LogNGal"] = np.log1p(c["NGal"])

    # Cosine / CSLS / Jaccard gaps
    c["CosGap"] = c["QTop1Cos"] - c["CosMax"]
    c["CSLSGap"] = c["QTop1CSLS"] - c["CSLSMax"]
    c["JacGap"] = c["QTop1Jac"] - c["JacMax"]
    c["SameGap"] = c["CosMax"] - c["SameSideMax"]
    c["OppGap"] = c["CosMax"] - c["OppSideMax"]
    c["HasSameSide"] = c["SameSideMax"].notna().astype(np.int8)

    c["QRawTop1"] = c.groupby("QKey")["RawCosMax"].transform("max")
    c["QWPTop1"] = c.groupby("QKey")["WPCosMax"].transform("max")
    c["RawGap"] = c["QRawTop1"] - c["RawCosMax"]
    c["WPGap"] = c["QWPTop1"] - c["WPCosMax"]
    c["QKnnTop"] = c.groupby("QKey")["Knn15"].transform("max")

    # Best-image raw cosine
    q_r = c["Query"].map(ROW_MAP).to_numpy()
    b_r = c["BestImgs"].str[0].map(ROW_MAP).to_numpy()
    c["RawCosBest"] = (F_UNIT[q_r] * F_UNIT[b_r]).sum(axis=1)

    c["VRel"] = vrel
    c["SpeciesC"] = c["Query"].map(lambda q: SPECIES_LIST.index(sp_map[q])).astype(np.int8)
    c["QSide"] = c["Query"].map(lambda q: SIDE_CODE[get_side(vp_map[q])]).astype(np.int8)
    return c

t0 = time.time()
V_FEAT = add_rich_features(V_ALL, SIFT_CACHE, NKP_MAP, VP_MAP, SP_MAP)
T_FEAT = add_rich_features(CAND_TEST, SIFT_CACHE, NKP_MAP, VP_MAP, SP_MAP)
print(f"Engineered features in {time.time() - t0:.1f}s | V_FEAT: {V_FEAT.shape} | T_FEAT: {T_FEAT.shape}")